# Métodos de Monte Carlo

Este Colab foi preparado para que você implemente um algoritmo de Monte Carlo em um ambiente Gridworld 4x3.

Você irá:

1. **Implementar o algoritmo de first-visit Monte Carlo control**: Preencher as partes faltando na classe `MonteCarloAgent` para inicializar o agente, atualizar estimativas a partir de uma trajetória num episódio e treinar o agente, gerando as trajetórias.
3. **Visualizar os resultados**: Utilizar as ferramentas de visualização fornecidas para inspecionar os valores dos estados e a política resultante.

Ao final, você terá uma implementação funcional de um agente de Monte Carlo e uma compreensão de como ele pode ser aplicado para encontrar a política ótima em um ambiente com dinâmica desconhecida.

## Grid 4x3

O código abaixo deve ser preenchido com a implementação do grid 4x3, com as mesmas especificações do Colab de [MDPs](https://colab.research.google.com/drive/1iNG9EX1-piXQvmN4-wQzncBv4EX7-yZV?usp=sharing). Portanto, você pode/deve aproveitar o código já feito. Não há mais aquelas funções adicionais para permitir que o ambiente seja "consultado" para execução de métodos de programação dinâmica, porque agora o agente não vai conhecer a dinâmica do ambiente. Ele irá interagir somente pela API padrão do gymnasium.


### Especificação do GridWorld (igual à tarefa de MDPs)

A célula abaixo tem a especificação do grid 4x3, igual no colab de MDPs. Pode pular se você já tiver feito.



Implemente o ambiente do grid 4x3, preenchendo as células abaixo. Você deve permitir ao usuário especificar a recompensa de cada passo (padrão = -0.04), a probabilidade de 'escorregar' (padrão = 0.2) e o numero máximo de passos antes de encerrar o episódio. O espaço de ações deve ser discreto, com 4 ações (0=UP, 1=RIGHT, 2=DOWN, 3=LEFT), e o de estados também será discreto, com 12 estados (mesmo o estado 'parede' pode ser considerado nesta contagem). A convenção para numeração dos estados é (G=goal, #=parede, P=pit/buraco,S=start):
```
y=2    +----+----+----+----+
       |  8 |  9 | 10 | 11G|
       +----+----+----+----+
y=1    |  4 |  5#|  6 |  7P|
       +----+----+----+----+
y=0    |  0S|  1 |  2 |  3 |
       +----+----+----+----+
        x=0   x=1   x=2   x=3
```

Note que há métodos para converter a numeração de estados para as coordenadas x,y

## Código do GridWorld 4x3

In [26]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from typing import Optional,Iterable,Tuple
import matplotlib.pyplot as plt


class GridWorld4x3(gym.Env):

    metadata = {"render_modes": ["ansi"]}

    def __init__(self, reward_step=-0.04, slip=0.2,
                 max_steps=1000, seed=None, render_mode=None):

        super().__init__()

        self.nrows = 3
        self.ncols = 4

        self.reward_step = reward_step
        self.slip = slip
        self.max_steps = max_steps

        self.start_pos = (0,0)
        self.wall_pos = (1,1)
        self.pit_pos = (3,1)
        self.goal_pos = (3,2)

        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Discrete(12)

        self.render_mode = render_mode
        self.pos = self.start_pos
        self.steps = 0

        self.reset(seed=seed)


    def pos_to_state(self,pos):
        x,y = pos
        return y*self.ncols+x


    def state_to_pos(self,s):
        return (s % self.ncols, s // self.ncols)


    def is_terminal(self,pos):
        return pos == self.goal_pos or pos == self.pit_pos


    def reset(self, *, seed=None, options=None):

        super().reset(seed=seed)

        self.pos = self.start_pos
        self.steps = 0

        return self.pos_to_state(self.pos), {}


    def step(self, action):

        self.steps += 1

        if self.np_random.random() < self.slip:
            action = int(self.np_random.integers(4))

        moves = {
            0:(0,1),
            1:(1,0),
            2:(0,-1),
            3:(-1,0)
        }

        dx,dy = moves[action]

        new_pos = (
            self.pos[0]+dx,
            self.pos[1]+dy
        )

        if (
            new_pos[0] < 0 or
            new_pos[0] >= self.ncols or
            new_pos[1] < 0 or
            new_pos[1] >= self.nrows or
            new_pos == self.wall_pos
        ):
            new_pos = self.pos


        self.pos = new_pos

        terminated = self.is_terminal(self.pos)
        truncated = self.steps >= self.max_steps

        reward = self.reward_step

        if self.pos == self.goal_pos:
            reward = 1

        elif self.pos == self.pit_pos:
            reward = -1


        return (
            self.pos_to_state(self.pos),
            reward,
            terminated,
            truncated,
            {}
        )


    def render(self, mode="ansi"):

        grid = [[" " for _ in range(4)] for _ in range(3)]

        grid[2][3] = "G"
        grid[1][3] = "P"
        grid[1][1] = "#"

        x,y = self.pos
        grid[y][x] = "A"

        return "\n".join(
            "|"+"|".join(row)+"|"
            for row in reversed(grid)
        )

## Codigo pra testar o Env

A célula abaixo faz o teste básico pra verificar que o ambiente respeita a interface gymnasium

In [27]:
# Deve continuar passando nos testes do Gymnasium
from gymnasium.utils.env_checker import check_env

# Criar uma instância do ambiente
env = GridWorld4x3()

# This will catch many common issues
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")

Environment passes all checks!


/usr/local/lib/python3.13/dist-packages/gymnasium/utils/env_checker.py:440: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantiating the environment through `gymnasium.make`
  logger.warn(


## Tarefa: Agente de Monte Carlo

Agora você irá implementar o algoritmo de first-visit Monte Carlo control. Você irá trabalhar na classe `MonteCarloAgent` fornecida abaixo. Seu objetivo é preencher as partes faltando para que o agente seja capaz de escolher ações e atualizar estimativas de valor para encontrar a política ótima em um ambiente. No caso deste enunciado será o grid 4x3.

Você irá:

1.  **Implementar a inicialização (`__init__`)**: Prepare o agente para manter o registro dos valores-Q e os retornos para cada par estado-ação visitado durante sua "vida".
2.  **Implementar `greedy_action(state)`**: Preencha este método para retornar a ação gulosa para um dado estado, com base nos valores-Q correntes.
3.  **Implementar `updateQ(episode)`**: Complete este método para receber um episódio completo e atualizar os valores-Q usando a abordagem de First-Visit Monte Carlo. Esta função não deve resetar o agente. Você deve "continuar" a atualização a partir dos valores-Q atuais. Você pode testar esta função independente da `train`, já que esta recebe um episódio pronto.
4.  **Implementar `train(n_episodes)`**: Desenvolva este método para executar múltiplos episódios utilizando uma política epsilon-gulosa, coletando os dados de cada episódio e utilizando a função `updateQ` para treinar o agente. Esta função também não deve resetar o agente. Você deve "continuar" o treino de onde tiver parado.

Ao final, você terá um agente capaz de aprender a política ótima do ambiente Gridworld 4x3 utilizando o algoritmo de Monte Carlo first-visit.

In [28]:
import numpy as np
import gymnasium as gym
from collections import defaultdict


class MonteCarloAgent:

    def __init__(self, env, gamma=0.9, epsilon=0.1):

        self.env = env
        self.gamma = gamma
        self.epsilon = epsilon

        self.Q_values = defaultdict(float)
        self.returns = defaultdict(list)


    def Q(self,state,action):

        return self.Q_values[(state,action)]


    def V(self,state):

        return max(
            self.Q(state,a)
            for a in range(self.env.action_space.n)
        )


    def greedy_action(self,state):

        values = [
            self.Q(state,a)
            for a in range(self.env.action_space.n)
        ]

        return int(np.argmax(values))


    def updateQ(self,episode):

        visited = set()
        G = 0

        returns = [0]*len(episode)

        for t in reversed(range(len(episode))):

            state,action,reward = episode[t]

            G = reward + self.gamma*G

            returns[t] = G


        for t,(state,action,reward) in enumerate(episode):

            pair = (state,action)

            if pair not in visited:

                visited.add(pair)

                self.returns[pair].append(
                    returns[t]
                )

                self.Q_values[pair] = np.mean(
                    self.returns[pair]
                )


    def train(self,n_episodes):

        for _ in range(n_episodes):

            state,_ = self.env.reset()

            episode=[]

            terminated=False
            truncated=False


            while not (terminated or truncated):

                if np.random.random() < self.epsilon:

                    action = self.env.action_space.sample()

                else:

                    action = self.greedy_action(state)


                next_state,reward,terminated,truncated,_ = self.env.step(action)

                episode.append(
                    (state,action,reward)
                )

                state = next_state


            self.updateQ(episode)


## Utilitário para visualizar o agente

O mesmo da tarefa anterior, porém "enxergando" um pouco menos dos detalhes internos do env

### Código do utilitário

In [36]:
class AgentVisualizer:

    def __init__(self, agent, env):
        """
        agent: objeto que implementa V(s), Q(s,a) e greedy_action(s)
        env: GridWorld4x3
        """

        self.agent = agent
        self.env = env

        self.action_to_str = {
            0: "↑",
            1: "→",
            2: "↓",
            3: "←"
        }

        self.wall_s = self.env.pos_to_state(
            self.env.wall_pos
        )

        self.start_s = self.env.pos_to_state(
            self.env.start_pos
        )

        self.goal_s = self.env.pos_to_state(
            self.env.goal_pos
        )

        self.pit_s = self.env.pos_to_state(
            self.env.pit_pos
        )


    # -----------------------
    # Política
    # -----------------------

    def print_policy(self):

        rows, cols = 3, 4

        horiz = "+" + "+".join(
            ["------"] * cols
        ) + "+"

        for y in reversed(range(rows)):

            print(horiz)

            cells = []

            for x in range(cols):

                s = self.env.pos_to_state((x,y))

                if s == self.wall_s:

                    content = "##"

                elif s == self.goal_s:

                    content = " G "

                elif s == self.pit_s:

                    content = " P "

                else:

                    a = self.agent.greedy_action(s)

                    arrow = self.action_to_str.get(
                        a,
                        "?"
                    )

                    if s == self.start_s:
                        content = f"S{arrow}"
                    else:
                        content = arrow

                cells.append(
                    f"{content:^6}"
                )

            print(
                "|" +
                "|".join(cells) +
                "|"
            )

        print(horiz)



    # -----------------------
    # Valores V(s)
    # -----------------------

    def print_values(self):

        rows, cols = 3,4

        horiz = "+" + "+".join(
            ["--------"] * cols
        ) + "+"

        for y in reversed(range(rows)):

            print(horiz)

            cells = []

            for x in range(cols):

                s = self.env.pos_to_state((x,y))

                if s == self.wall_s:

                    content = "####"

                else:

                    v = self.agent.V(s)

                    if s == self.goal_s:

                        content = f"G({v:.2f})"

                    elif s == self.pit_s:

                        content = f"P({v:.2f})"

                    else:

                        content = f"{v:6.2f}"


                cells.append(
                    f"{content:^8}"
                )

            print(
                "|" +
                "|".join(cells) +
                "|"
            )

        print(horiz)



    # -----------------------
    # Q-values
    # -----------------------

    def print_qvalues(self):

        rows, cols = 3,4

        horiz = "+" + "+".join(
            ["---------------"] * cols
        ) + "+"


        for y in reversed(range(rows)):

            print(horiz)

            line1 = []
            line2 = []
            line3 = []


            for x in range(cols):

                s = self.env.pos_to_state((x,y))


                if s == self.wall_s:

                    c1 = "###############"
                    c2 = "###############"
                    c3 = "###############"


                else:

                    qvals = [
                        self.agent.Q(s,a)
                        for a in range(4)
                    ]


                    up = f"↑:{qvals[0]:.2f}"
                    left = f"←:{qvals[3]:.2f}"
                    right = f"→:{qvals[1]:.2f}"
                    down = f"↓:{qvals[2]:.2f}"


                    c1 = f"{up:^15}"

                    c2 = (
                        f"{left:<7}"
                        f"{right:>8}"
                    )

                    c3 = f"{down:^15}"


                line1.append(c1)
                line2.append(c2)
                line3.append(c3)



            print(
                "|" +
                "|".join(line1) +
                "|"
            )

            print(
                "|" +
                "|".join(line2) +
                "|"
            )

            print(
                "|" +
                "|".join(line3) +
                "|"
            )


        print(horiz)

### Exemplo de uso - 1 episódio simples

Ao executar o código a seguir, a saída esperada por ter executado as ações (acima, abaixo, acima, acima, direita, direita, direita) com reward_step=-0.1, slip=0 e gamma = 1 é como abaixo.



```
Teste. Parametros: reward_step=-0.1, slip=0, gamma=1
=== Política ===
+------+------+------+------+
|  →   |  →   |  →   |  G   |
+------+------+------+------+
|  ↑   |  ##  |  ↑   |  P   |
+------+------+------+------+
|  S↑  |  ↑   |  ↑   |  ↑   |
+------+------+------+------+

=== V(s) ===
+--------+--------+--------+--------+
|   0.80 |   0.90 |   1.00 |G(0.00) |
+--------+--------+--------+--------+
|   0.70 |  ####  |   0.00 |P(0.00) |
+--------+--------+--------+--------+
|   0.40 |   0.00 |   0.00 |   0.00 |
+--------+--------+--------+--------+

=== Q(s,a) ===
+---------------+---------------+---------------+---------------+
|    ↑:0.00     |    ↑:0.00     |    ↑:0.00     |    ↑:0.00     |
|←:0.00   →:0.80|←:0.00   →:0.90|←:0.00   →:1.00|←:0.00   →:0.00|
|    ↓:0.00     |    ↓:0.00     |    ↓:0.00     |    ↓:0.00     |
+---------------+---------------+---------------+---------------+
|    ↑:0.70     |###############|    ↑:0.00     |    ↑:0.00     |
|←:0.00   →:0.00|###############|←:0.00   →:0.00|←:0.00   →:0.00|
|    ↓:0.50     |###############|    ↓:0.00     |    ↓:0.00     |
+---------------+---------------+---------------+---------------+
|    ↑:0.40     |    ↑:0.00     |    ↑:0.00     |    ↑:0.00     |
|←:0.00   →:0.00|←:0.00   →:0.00|←:0.00   →:0.00|←:0.00   →:0.00|
|    ↓:0.00     |    ↓:0.00     |    ↓:0.00     |    ↓:0.00     |
+---------------+---------------+---------------+---------------+
```

In [37]:
env = GridWorld4x3()

agent = MonteCarloAgent(
    env,
    gamma=0.9,
    epsilon=0.2
)

viz = AgentVisualizer(agent,env)

agent.train(1000)

print("=== Política final ===")
viz.print_policy()

print("=== Valores V ===")
viz.print_values()

=== Política final ===
+------+------+------+------+
|  →   |  →   |  →   |  G   |
+------+------+------+------+
|  ↑   |  ##  |  ↑   |  P   |
+------+------+------+------+
|  S→  |  →   |  ↑   |  ←   |
+------+------+------+------+
=== Valores V ===
+--------+--------+--------+--------+
|   0.50 |   0.69 |   0.93 |G(0.00) |
+--------+--------+--------+--------+
|   0.31 |  ####  |   0.62 |P(0.00) |
+--------+--------+--------+--------+
|   0.17 |   0.27 |   0.40 |   0.19 |
+--------+--------+--------+--------+


### Exemplo de uso - outro episódio simples

Ao executar o código a seguir, a saída esperada por ter executado as ações (acima, abaixo, acima, acima, direita, direita, direita) com reward_step=-0.1, slip=0 e gamma = 0.9 é como abaixo.



```
Teste. Parametros: reward_step=-0.1, slip=0, gamma=0.9
=== Política ===
+------+------+------+------+
|  →   |  →   |  →   |  G   |
+------+------+------+------+
|  ↑   |  ##  |  ↑   |  P   |
+------+------+------+------+
|  S↑  |  ↑   |  ↑   |  ↑   |
+------+------+------+------+

=== V(s) ===
+--------+--------+--------+--------+
|   0.62 |   0.80 |   1.00 |G(0.00) |
+--------+--------+--------+--------+
|   0.46 |  ####  |   0.00 |P(0.00) |
+--------+--------+--------+--------+
|   0.06 |   0.00 |   0.00 |   0.00 |
+--------+--------+--------+--------+

=== Q(s,a) ===
+---------------+---------------+---------------+---------------+
|    ↑:0.00     |    ↑:0.00     |    ↑:0.00     |    ↑:0.00     |
|←:0.00   →:0.62|←:0.00   →:0.80|←:0.00   →:1.00|←:0.00   →:0.00|
|    ↓:0.00     |    ↓:0.00     |    ↓:0.00     |    ↓:0.00     |
+---------------+---------------+---------------+---------------+
|    ↑:0.46     |###############|    ↑:0.00     |    ↑:0.00     |
|←:0.00   →:0.00|###############|←:0.00   →:0.00|←:0.00   →:0.00|
|    ↓:0.18     |###############|    ↓:0.00     |    ↓:0.00     |
+---------------+---------------+---------------+---------------+
|    ↑:0.06     |    ↑:0.00     |    ↑:0.00     |    ↑:0.00     |
|←:0.00   →:0.00|←:0.00   →:0.00|←:0.00   →:0.00|←:0.00   →:0.00|
|    ↓:0.00     |    ↓:0.00     |    ↓:0.00     |    ↓:0.00     |
+---------------+---------------+---------------+---------------+
```

In [38]:
env = GridWorld4x3(reward_step=-0.1, slip=0) # grid deterministico
agent = MonteCarloAgent(env, gamma=0.9) # agente sem desconto
viz = AgentVisualizer(agent, env)

#0=UP, 1=RIGHT, 2=DOWN, 3=LEFT

# politica simples: vai pra cima (0), baixo (2), sobe tudo(0), depois tudo pra direita (1)
simple_episode = [
    (0, 0, -0.1), #start, up
    (4, 2, -0.1), #2,1 - down
    (0, 0, -0.1), #start, up
    (4, 0, -0.1), #2,1 - up
    (8, 1, -0.1), #3,1 - right
    (9, 1, -0.1), #3,2 - right
    (10, 1, +1), #3,3 - right (done)
]

print(f"\nTeste. Parametros: reward_step={env.reward_step}, slip={env.slip}, gamma={agent.gamma}")
agent.updateQ(simple_episode)

print("=== Política ===")
viz.print_policy()

print("\n=== V(s) ===")
viz.print_values()

print("\n=== Q(s,a) ===")
viz.print_qvalues()

env.close()


Teste. Parametros: reward_step=-0.1, slip=0, gamma=0.9
=== Política ===
+------+------+------+------+
|  →   |  →   |  →   |  G   |
+------+------+------+------+
|  ↑   |  ##  |  ↑   |  P   |
+------+------+------+------+
|  S↑  |  ↑   |  ↑   |  ↑   |
+------+------+------+------+

=== V(s) ===
+--------+--------+--------+--------+
|   0.62 |   0.80 |   1.00 |G(0.00) |
+--------+--------+--------+--------+
|   0.46 |  ####  |   0.00 |P(0.00) |
+--------+--------+--------+--------+
|   0.06 |   0.00 |   0.00 |   0.00 |
+--------+--------+--------+--------+

=== Q(s,a) ===
+---------------+---------------+---------------+---------------+
|    ↑:0.00     |    ↑:0.00     |    ↑:0.00     |    ↑:0.00     |
|←:0.00   →:0.62|←:0.00   →:0.80|←:0.00   →:1.00|←:0.00   →:0.00|
|    ↓:0.00     |    ↓:0.00     |    ↓:0.00     |    ↓:0.00     |
+---------------+---------------+---------------+---------------+
|    ↑:0.46     |###############|    ↑:0.00     |    ↑:0.00     |
|←:0.00   →:0.00|########

### Treino (1000 episódios) e visualização do agente

Com 1000 episódios de treino, o agente deve poder encontrar a trajetória correta a partir do estado inicial. Porém, não se assuste se a ação para alguns estados menos visitados não for encontrada.

In [39]:
env = GridWorld4x3() # grid com parâmetros padrão
agent = MonteCarloAgent(env, gamma=0.9, epsilon=0.2) # agente com desconto e epsilon-greedy
viz = AgentVisualizer(agent, env)

print("Treinando agente por 1000 episódios...")
agent.train(1000)
print("Treino concluído.")

print("\n=== Política final ===")
viz.print_policy()

print("\n=== V(s) final ===")
viz.print_values()

print("\n=== Q(s,a) final ===")
viz.print_qvalues()

env.close()

Treinando agente por 1000 episódios...
Treino concluído.

=== Política final ===
+------+------+------+------+
|  →   |  →   |  →   |  G   |
+------+------+------+------+
|  ↑   |  ##  |  ↑   |  P   |
+------+------+------+------+
|  S↑  |  ←   |  ↑   |  ↓   |
+------+------+------+------+

=== V(s) final ===
+--------+--------+--------+--------+
|   0.57 |   0.74 |   0.95 |G(0.00) |
+--------+--------+--------+--------+
|   0.43 |  ####  |   0.65 |P(0.00) |
+--------+--------+--------+--------+
|   0.30 |   0.26 |   0.36 |   0.00 |
+--------+--------+--------+--------+

=== Q(s,a) final ===
+---------------+---------------+---------------+---------------+
|    ↑:0.47     |    ↑:0.60     |    ↑:0.75     |    ↑:0.00     |
|←:0.44   →:0.57|←:0.48   →:0.74|←:0.60   →:0.95|←:0.00   →:0.00|
|    ↓:0.36     |    ↓:0.62     |    ↓:0.58     |    ↓:0.00     |
+---------------+---------------+---------------+---------------+
|    ↑:0.43     |###############|    ↑:0.65     |    ↑:0.00     |
|←:0.

## Bônus: explore sua implementação

Teste diversos parâmetros para verificar seu efeito nos valores e na política. Você pode comparar seus resultados daqui com os da sua implementação de iteração de valor, para ver como o agente de Monte Carlo tem mais "dificuldade" de obter a política ótima